# 🌾 Crop Yield Prediction Based on Weather Data

**Objective:** Predict crop yield (tons/hectare) using weather and soil features.

**Approach:**
- Synthetic dataset simulating real agricultural conditions
- Exploratory Data Analysis with rich visualizations
- Feature Engineering (interaction terms, seasonal indicators)
- Regression Models: Linear, Random Forest, Gradient Boosting
- Evaluation with MAE, RMSE, R²

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Data Generation

We create a realistic synthetic dataset. Replace with your own CSV:
```python
# df = pd.read_csv('/kaggle/input/crop-yield-prediction-dataset/yield_df.csv')
```

In [2]:
d = pd.read_csv('/kaggle/input/datasets/patelris/crop-yield-prediction-dataset/yield.csv')
d

,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value
0,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1961,1961,hg/ha,14000
1,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1962,1962,hg/ha,14000
2,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1963,1963,hg/ha,14260
3,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1964,1964,hg/ha,14257
4,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1965,1965,hg/ha,14400
...,...,...,...,...,...,...,...,...,...,...,...,...
56712,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2012,2012,hg/ha,24420
56713,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2013,2013,hg/ha,22888
56714,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2014,2014,hg/ha,21357
56715,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2015,2015,hg/ha,19826


In [3]:
print('--- Basic Statistics ---')
df.describe().round(2)

--- Basic Statistics ---


NameError: name 'df' is not defined

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Yield distribution
sns.histplot(df['Yield_ton_ha'], kde=True, ax=axes[0, 0], color='forestgreen', bins=40)
axes[0, 0].set_title('Yield Distribution')
axes[0, 0].set_xlabel('Yield (ton/ha)')

# Yield by crop
sns.boxplot(data=df, x='Crop', y='Yield_ton_ha', ax=axes[0, 1], palette='Set2', hue='Crop', legend=False)
axes[0, 1].set_title('Yield by Crop Type')

# Rainfall vs Yield
sns.scatterplot(data=df, x='Avg_Rainfall_mm', y='Yield_ton_ha', hue='Crop',
                alpha=0.5, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Rainfall vs Yield')
axes[1, 0].legend(fontsize=8)

# Temperature vs Yield
sns.scatterplot(data=df, x='Avg_Temp_C', y='Yield_ton_ha', hue='Crop',
                alpha=0.5, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Temperature vs Yield')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Yield by season
sns.violinplot(data=df, x='Season', y='Yield_ton_ha', ax=axes[0], palette='pastel', hue='Season', legend=False)
axes[0].set_title('Yield Distribution by Season')

# Yield by soil type
sns.boxplot(data=df, x='Soil_Type', y='Yield_ton_ha', ax=axes[1], palette='pastel', hue='Soil_Type', legend=False)
axes[1].set_title('Yield by Soil Type')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='RdYlGn', center=0, fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
data = df.copy()

# Interaction features
data['Temp_x_Rainfall'] = data['Avg_Temp_C'] * data['Avg_Rainfall_mm'] / 1000
data['NPK_total'] = data['Nitrogen_kg_ha'] + data['Phosphorus_kg_ha'] + data['Potassium_kg_ha']
data['Humidity_x_Sunshine'] = data['Humidity_pct'] * data['Sunshine_Hrs'] / 100
data['Temp_deviation'] = np.abs(data['Avg_Temp_C'] - 27)

# Encode categoricals
le_crop = LabelEncoder()
le_season = LabelEncoder()
le_soil = LabelEncoder()

data['Crop_enc'] = le_crop.fit_transform(data['Crop'])
data['Season_enc'] = le_season.fit_transform(data['Season'])
data['Soil_enc'] = le_soil.fit_transform(data['Soil_Type'])

# One-hot encode for linear models
data = pd.get_dummies(data, columns=['Crop', 'Season', 'Soil_Type'], drop_first=True)

print(f'Processed shape: {data.shape}')
data.head()

In [ ]:
X = data.drop(columns=['Yield_ton_ha'])
y = data['Yield_ton_ha']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')
print(f'Yield stats — mean: {y.mean():.2f}, std: {y.std():.2f}')

## 4. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

results = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2')
    
    results[name] = {'model': model, 'y_pred': y_pred, 'mae': mae, 'rmse': rmse, 'r2': r2,
                     'cv_r2_mean': cv_r2.mean(), 'cv_r2_std': cv_r2.std()}
    
    print(f'\n--- {name} ---')
    print(f'MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | CV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

In [ ]:
# Model comparison bar chart
comp_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [r['mae'] for r in results.values()],
    'RMSE': [r['rmse'] for r in results.values()],
    'R²': [r['r2'] for r in results.values()]
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

for i, metric in enumerate(['MAE', 'RMSE', 'R²']):
    bars = axes[i].bar(comp_df['Model'], comp_df[metric], color=colors)
    axes[i].set_title(metric, fontsize=14, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=25)
    for bar, val in zip(bars, comp_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted for best model
best_name = max(results, key=lambda k: results[k]['r2'])
best_pred = results[best_name]['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, best_pred, alpha=0.4, c='teal', edgecolors='k', linewidth=0.3, s=20)
lims = [min(y_test.min(), best_pred.min()) - 0.5, max(y_test.max(), best_pred.max()) + 0.5]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual Yield (ton/ha)')
axes[0].set_ylabel('Predicted Yield (ton/ha)')
axes[0].set_title(f'Actual vs Predicted — {best_name}')
axes[0].legend()

# Residual distribution
residuals = y_test.values - best_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='coral', bins=40)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title(f'Residual Distribution — {best_name}')
axes[1].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

print(f'Residual mean: {residuals.mean():.4f}, std: {residuals.std():.4f}')

## 5. Feature Importance

In [ ]:
best_model = results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
else:
    imp = pd.Series(np.abs(best_model.coef_), index=X.columns).sort_values(ascending=True)

imp.tail(15).plot(kind='barh', color='seagreen', figsize=(10, 6))
plt.title(f'Top 15 Features — {best_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Prediction Demo

In [ ]:
# Predict on a few test samples
sample_idx = np.random.choice(len(X_test), 8, replace=False)
sample_X = X_test_scaled[sample_idx]
sample_actual = y_test.values[sample_idx]
sample_pred = results[best_name]['model'].predict(sample_X)

demo_df = pd.DataFrame({
    'Actual Yield': np.round(sample_actual, 2),
    'Predicted Yield': np.round(sample_pred, 2),
    'Error': np.round(sample_actual - sample_pred, 2)
})
demo_df

## 7. Summary

**Key Findings:**
- Crop type is the dominant predictor of yield (different base yields)
- Nitrogen content and rainfall are the top weather/soil features
- Temperature deviation from 27°C negatively impacts yield
- Ensemble methods (RF, GBR) significantly outperform linear models

**Next Steps:**
- Incorporate real-world datasets (e.g., FAO, India Open Data)
- Add temporal features (year-over-year trends)
- Experiment with XGBoost / LightGBM
- Build a deployment pipeline with Flask or Streamlit

---
*Notebook by Data Science Pipeline | Kaggle*